<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/10_ETL_Multimedia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir S10 en Google Colab"></a>

# Sesión 10 · ETL multimodal: de imágenes, audio y video a datos trazables

**Universidad Central · Maestría en Analítica de Datos · Big Data 2026-2S**

[Presentación interactiva + laboratorio](https://jazaineam1.github.io/BigData2026/Presentaciones/s10-etl-multimedia.html)

## Pregunta profesional

Una noticia de El Tiempo puede traer texto, imágenes, audio y video. Hasta ahora sabemos que esos activos existen; hoy construiremos una tubería que permita **identificarlos, inspeccionarlos, transformarlos, validarlos y cargarlos sin perder su procedencia**.

**Producto:** `assets.jsonl`, `assets.parquet` y `manifest_s10.json` generados por tu propia ejecución.

## Mapa de trabajo

1. Recuperar el caso real de El Tiempo y cambiar el grano de noticia a **asset**.
2. Conservar RAW e identificarlo con SHA-256 y tipo.
3. Inspeccionar imagen y crear un thumbnail sin sobrescribir el original.
4. Inspeccionar audio y canonicalizarlo a WAV 16 kHz mono.
5. Inspeccionar video, separar su estructura y muestrear frames.
6. Construir un manifest común.
7. Ejecutar quality gates objetivos.
8. Exportar JSONL y Parquet y dejar el puente listo para S11.

**Regla de procedencia.** El dataset y la URL del artículo son reales. Cuando el dataset actual no conserva un binario estable de audio/video, el cuaderno crea un `fixture_derivado` reproducible y lo registra explícitamente como tal.

## Vocabulario mínimo

| Término | Significa aquí | No significa |
|---|---|---|
| asset | archivo digital procesable | noticia completa |
| RAW | archivo recibido sin sobrescribir | archivo “sucio” |
| metadata | datos que describen el activo | interpretación semántica |
| hash | huella de los bytes | similitud de significado |
| container | estructura que contiene streams | codec |
| codec | forma de codificar/decodificar un stream | extensión |
| stream | flujo interno de audio/video | archivo completo |
| canonicalización | convertir entradas heterogéneas a una forma acordada | mejorar significado |
| derived asset | archivo producido desde RAW | reemplazo del original |
| provenance | relación entre origen, proceso y resultado | comentario libre |

## Caso real y procedencia

El dataset `Datos/noticias_eltiempo_2026-08.json` fue construido por `utils/build_eltiempo_dataset.py` a partir del sitemap mensual y del feed JSON público por artículo. La recolección fue docente, con pausa y User-Agent identificado.

El archivo de metadata registra **126 noticias**, las 126 con arreglo de imágenes, las 126 con bandera de audio y 8 con bandera de video.

**OJO.** El dataset curado conserva URL/crédito de imágenes y banderas de audio/video; no conserva todos los binarios. Por eso este laboratorio distingue siempre entre `eltiempo_real` y `fixture_derivado`.

In [ ]:
from pathlib import Path
import hashlib, json, math, mimetypes, shutil, subprocess, urllib.request
import pandas as pd
from PIL import Image, ImageDraw

ROOT = Path("work/s10")
RAW = ROOT / "raw"
DERIVED = ROOT / "derived"
OUT = ROOT / "outputs"
for p in [RAW/"image", RAW/"audio", RAW/"video", DERIVED/"thumbnails", DERIVED/"audio", DERIVED/"frames", OUT]:
    p.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/noticias_eltiempo_2026-08.json"
with urllib.request.urlopen(DATA_URL, timeout=30) as r:
    noticias = json.loads(r.read().decode("utf-8"))

candidatos = [n for n in noticias if n.get("imagenes")]
print("Noticias cargadas:", len(noticias))
print("Con al menos una imagen:", len(candidatos))
print("Carpetas de trabajo:", ROOT)

In [ ]:
INDICE_ARTICULO = 7 #@param {type:"integer"}

noticia = candidatos[INDICE_ARTICULO % len(candidatos)]
imagen_meta = noticia["imagenes"][0]

print("article_id:", noticia["_id"])
print("titulo:", noticia["titulo"])
print("url artículo:", noticia["url"])
print("url imagen:", imagen_meta.get("url"))
print("crédito:", imagen_meta.get("credito"))

### Leamos la selección real

**Cómo se lee.** El índice numérico elige una noticia dentro de la colección real versionada; el `article_id` y las URLs vienen del dataset.

**Qué nos dice.** Dos estudiantes pueden ejecutar la misma tubería sobre artículos distintos sin redactar respuestas abiertas.

**Qué NO permite concluir todavía.** La URL no demuestra que el binario siga disponible ni que su contenido haya sido descargado correctamente.

**Error común.** Confundir “tengo una URL” con “tengo un activo reproducible”.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def ffprobe_json(path):
    if shutil.which("ffprobe") is None:
        raise RuntimeError("ffprobe no está disponible en este runtime")
    cmd = ["ffprobe", "-v", "error", "-show_format", "-show_streams", "-of", "json", str(path)]
    return json.loads(subprocess.check_output(cmd, text=True))

def run_ffmpeg(args):
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg no está disponible en este runtime")
    subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", *args], check=True)

print("ffmpeg:", shutil.which("ffmpeg"))
print("ffprobe:", shutil.which("ffprobe"))

## LAB 1 · Imagen — RAW real cuando está disponible

La tubería intenta **una sola descarga** del primer activo de imagen de la noticia seleccionada. Si la fuente no responde, crea un PNG local con el título del artículo para que el laboratorio continúe.

Ese respaldo queda marcado como `fixture_derivado`. No se presenta como imagen editorial real.

In [ ]:
img_url = imagen_meta.get("url")
raw_img = RAW / "image" / f"article_{noticia['_id']}.jpg"
origin_kind = "eltiempo_real"
download_error = None

try:
    req = urllib.request.Request(img_url, headers={"User-Agent": "BigData-UCentral-lab/1.0"})
    with urllib.request.urlopen(req, timeout=15) as r:
        payload = r.read()
    raw_img.write_bytes(payload)
    with Image.open(raw_img) as test:
        test.verify()
except Exception as exc:
    download_error = type(exc).__name__
    origin_kind = "fixture_derivado"
    raw_img = RAW / "image" / f"article_{noticia['_id']}_fixture.png"
    canvas = Image.new("RGB", (1280, 720), "#122253")
    draw = ImageDraw.Draw(canvas)
    draw.text((70, 90), "FIXTURE DIDÁCTICO · NO ES FOTO EDITORIAL", fill="white")
    titulo = (noticia["titulo"] or "")[:110]
    for i in range(0, len(titulo), 42):
        draw.text((70, 190 + (i//42)*55), titulo[i:i+42], fill="#ffd9c6")
    canvas.save(raw_img)

with Image.open(raw_img) as im:
    image_info = {"format": im.format, "width": im.width, "height": im.height, "mode": im.mode}

print("origin_kind:", origin_kind)
print("download_error:", download_error)
print("path:", raw_img)
print("sha256:", sha256_file(raw_img))
print("image_info:", image_info)

### Leamos la imagen

**Cómo se lee.** `origin_kind` diferencia un binario descargado de la fuente de un fixture de respaldo. `sha256` identifica exactamente los bytes guardados.

**Qué nos dice.** Ya tenemos un RAW auditable y metadata técnica mínima.

**Qué NO permite concluir todavía.** Dimensiones, formato y hash no describen qué personas, objetos o hechos aparecen en la imagen.

**Error común.** Ocultar el fallback y seguir hablando como si el archivo hubiera venido de la fuente. Eso rompe provenance.

In [ ]:
thumb = DERIVED / "thumbnails" / f"{raw_img.stem}_640.jpg"
with Image.open(raw_img).convert("RGB") as im:
    im.thumbnail((640, 640))
    im.save(thumb, "JPEG", quality=88)

with Image.open(thumb) as im:
    thumb_info = {"width": im.width, "height": im.height, "format": im.format}

print("RAW:", raw_img)
print("DERIVED:", thumb)
print("thumbnail:", thumb_info)
print("raw sigue existiendo:", raw_img.exists())

### RAW ≠ derived

**Cómo se lee.** Existen dos rutas distintas: el original y el thumbnail.

**Qué nos dice.** Podemos crear una representación optimizada sin borrar evidencia.

**Qué NO permite concluir todavía.** Tener un thumbnail correcto no demuestra que todo el pipeline multimedia sea idempotente.

**Error común.** Guardar el thumbnail encima del RAW porque “se ve igual”.

## LAB 2 · Audio — canonicalización reproducible

El dataset real registra que la noticia puede tener audio, pero la versión curada no conserva su binario. Para enseñar el mecanismo sin inventar procedencia, generamos un tono de prueba **determinado por el article_id**.

El artículo es real; el audio se registra como `fixture_derivado`.

In [ ]:
audio_raw = RAW / "audio" / f"article_{noticia['_id']}_source.wav"
freq = 320 + (int(noticia["_id"]) % 360)

run_ffmpeg([
    "-f", "lavfi", "-i", f"sine=frequency={freq}:sample_rate=44100:duration=8",
    "-filter_complex", "[0:a]pan=stereo|c0=c0|c1=c0[a]",
    "-map", "[a]", "-c:a", "pcm_s16le", str(audio_raw)
])

audio_before = ffprobe_json(audio_raw)
astream = next(s for s in audio_before["streams"] if s["codec_type"] == "audio")
print("fixture:", audio_raw)
print("codec:", astream["codec_name"])
print("sample_rate:", astream["sample_rate"])
print("channels:", astream["channels"])
print("duration_s:", audio_before["format"]["duration"])

In [ ]:
audio_canon = DERIVED / "audio" / f"article_{noticia['_id']}_16k_mono.wav"
run_ffmpeg(["-i", str(audio_raw), "-ac", "1", "-ar", "16000", str(audio_canon)])

audio_after = ffprobe_json(audio_canon)
astream2 = next(s for s in audio_after["streams"] if s["codec_type"] == "audio")

print("ANTES:", astream["sample_rate"], "Hz ·", astream["channels"], "canales")
print("DESPUÉS:", astream2["sample_rate"], "Hz ·", astream2["channels"], "canal")
print("sha RAW:", sha256_file(audio_raw))
print("sha derived:", sha256_file(audio_canon))

### Leamos la canonicalización

**Cómo se lee.** Comparamos metadata antes y después de la transformación.

**Qué nos dice.** El derived cumple la interfaz acordada: 16 kHz, mono.

**Qué NO permite concluir todavía.** Canonicalizar no mejora automáticamente calidad ni reconocimiento de voz.

**Error común.** Ver dos hashes distintos y pensar que hubo corrupción: aquí esperamos bytes distintos porque hubo una transformación intencional.

## LAB 3 · Video — componer y reutilizar las tuberías

El fixture de video combina la imagen de trabajo con el audio anterior. Así podemos inspeccionar un container real y demostrar que contiene streams diferentes.

Después extraemos frames y aplicamos una regla de muestreo limitada por `MAX_FRAMES`.

In [ ]:
video_raw = RAW / "video" / f"article_{noticia['_id']}.mp4"
run_ffmpeg([
    "-loop", "1", "-i", str(raw_img),
    "-i", str(audio_raw),
    "-vf", "scale=1280:720:force_original_aspect_ratio=decrease,pad=1280:720:(ow-iw)/2:(oh-ih)/2",
    "-c:v", "libx264", "-tune", "stillimage", "-pix_fmt", "yuv420p",
    "-c:a", "aac", "-shortest", "-t", "8", str(video_raw)
])

video_meta = ffprobe_json(video_raw)
for s in video_meta["streams"]:
    print(s["codec_type"], "→", s.get("codec_name"),
          s.get("width", ""), s.get("height", ""),
          s.get("sample_rate", ""), s.get("channels", ""))
print("duration_s:", video_meta["format"]["duration"])
print("sha256:", sha256_file(video_raw))

### Leamos el container

**Cómo se lee.** `streams` muestra componentes internos del mismo archivo MP4.

**Qué nos dice.** Container y codec no son sinónimos: el container MP4 guarda un stream H.264 y otro AAC.

**Qué NO permite concluir todavía.** La existencia de dos streams no dice qué momentos del video son importantes.

**Error común.** Llamar “MP4” al codec de video.

In [ ]:
MAX_FRAMES = 8 #@param {type:"slider", min:4, max:12, step:1}

duration_s = float(video_meta["format"]["duration"])
interval_s = max(1.0, duration_s / MAX_FRAMES)
frames_dir = DERIVED / "frames"
for old in frames_dir.glob("frame_*.jpg"):
    old.unlink()

run_ffmpeg([
    "-i", str(video_raw),
    "-vf", f"fps=1/{interval_s},scale=640:-1",
    str(frames_dir / "frame_%03d.jpg")
])
frames = sorted(frames_dir.glob("frame_*.jpg"))

print("duration_s:", round(duration_s, 3))
print("MAX_FRAMES:", MAX_FRAMES)
print("interval_s:", round(interval_s, 3))
print("frames:", len(frames))
print("cumple límite:", len(frames) <= MAX_FRAMES)

### Leamos el sampling

**Cómo se lee.** El intervalo se deriva de duración y del límite elegido; el número final de frames se verifica.

**Qué nos dice.** El costo de procesamiento se controla con una regla explícita y reproducible.

**Qué NO permite concluir todavía.** Muestrear temporalmente no identifica automáticamente los momentos semánticamente más importantes.

**Error común.** Llamar “keyframes” a todos los frames muestreados.

## Load · un manifest común

Una fila del dataset final representa un **asset primario**. El registro conserva:

- `article_id` y URL de la noticia;
- `origin_kind`;
- ruta RAW, bytes y SHA-256;
- metadata técnica específica;
- rutas de derivados;
- parámetros de transformación.

La misma estructura puede ampliarse después con OCR, ASR o embeddings sin perder el original.

In [ ]:
def size_bytes(path):
    return Path(path).stat().st_size

vstream = next(s for s in video_meta["streams"] if s["codec_type"] == "video")
vaudio = next(s for s in video_meta["streams"] if s["codec_type"] == "audio")

records = [
    {
        "asset_id": sha256_file(raw_img),
        "article_id": int(noticia["_id"]),
        "media_type": "image",
        "origin_kind": origin_kind,
        "source_url": imagen_meta.get("url"),
        "raw_path": str(raw_img),
        "bytes": size_bytes(raw_img),
        "width": image_info["width"],
        "height": image_info["height"],
        "codec": image_info["format"],
        "derived": [str(thumb)],
    },
    {
        "asset_id": sha256_file(audio_raw),
        "article_id": int(noticia["_id"]),
        "media_type": "audio",
        "origin_kind": "fixture_derivado",
        "source_url": None,
        "raw_path": str(audio_raw),
        "bytes": size_bytes(audio_raw),
        "sample_rate": int(astream["sample_rate"]),
        "channels": int(astream["channels"]),
        "codec": astream["codec_name"],
        "derived": [str(audio_canon)],
    },
    {
        "asset_id": sha256_file(video_raw),
        "article_id": int(noticia["_id"]),
        "media_type": "video",
        "origin_kind": "fixture_derivado",
        "source_url": None,
        "raw_path": str(video_raw),
        "bytes": size_bytes(video_raw),
        "duration_s": float(video_meta["format"]["duration"]),
        "width": int(vstream["width"]),
        "height": int(vstream["height"]),
        "codec": vstream["codec_name"],
        "audio_codec": vaudio["codec_name"],
        "sampling_interval_s": interval_s,
        "derived": [str(x) for x in frames],
    },
]

df_assets = pd.DataFrame(records)
jsonl_path = OUT / "assets.jsonl"
parquet_path = OUT / "assets.parquet"
df_assets.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
df_assets.to_parquet(parquet_path, index=False)

manifest = {
    "pipeline": "s10_multimedia_etl",
    "version": 1,
    "article_id": int(noticia["_id"]),
    "article_url": noticia["url"],
    "input_assets": len(records),
    "by_media_type": df_assets["media_type"].value_counts().to_dict(),
    "origin_kinds": df_assets["origin_kind"].value_counts().to_dict(),
    "derived_frames": len(frames),
    "max_frames": MAX_FRAMES,
    "outputs": {"jsonl": str(jsonl_path), "parquet": str(parquet_path)},
}
manifest_path = OUT / "manifest_s10.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print(df_assets[["media_type","origin_kind","bytes","codec"]].to_string(index=False))
print("manifest:", manifest_path)

In [ ]:
checks = {
    "3 assets primarios": len(df_assets) == 3,
    "hashes SHA-256": df_assets["asset_id"].str.fullmatch(r"[0-9a-f]{64}").all(),
    "RAW existe": df_assets["raw_path"].map(lambda p: Path(p).exists()).all(),
    "imagen con dimensiones": image_info["width"] > 0 and image_info["height"] > 0,
    "audio canonical 16 kHz mono": int(astream2["sample_rate"]) == 16000 and int(astream2["channels"]) == 1,
    "video con dos tipos de stream": {"video","audio"} <= {s["codec_type"] for s in video_meta["streams"]},
    "sampling respeta límite": len(frames) <= MAX_FRAMES,
    "outputs existen": jsonl_path.exists() and parquet_path.exists() and manifest_path.exists(),
}

for name, ok in checks.items():
    print(("PASS" if ok else "FAIL"), "·", name)

assert all(checks.values()), "Hay un quality gate fallido"
print("QUALITY GATES: OK")

### Lectura final

**Cómo se lee.** Cada gate prueba una propiedad objetiva del pipeline y termina en PASS/FAIL.

**Qué nos dice.** La ejecución produjo tres activos primarios, preservó RAW, generó derivados trazables y dejó JSONL/Parquet/manifest coherentes.

**Qué NO permite concluir todavía.** Que el mismo pipeline escale a millones de activos ni que el costo sea aceptable.

**Error común.** Confundir “pasó en mi notebook” con “está listo para producción”. S11 empieza exactamente en esa diferencia.

In [ ]:
q_container = "Elige..." #@param ["Elige...", "Container", "Codec", "Hash"]
q_hash = "Elige..." #@param ["Elige...", "Detectar bytes idénticos", "Medir significado", "Encontrar keyframes"]
q_sampling = "Elige..." #@param ["Elige...", "Muestreo temporal", "Detección semántica de escenas", "OCR"]
q_raw = "Elige..." #@param ["Elige...", "Conservar RAW y escribir derived aparte", "Sobrescribir RAW para ahorrar espacio"]

answers = {
    "Container vs codec": q_container == "Container",
    "Uso del hash": q_hash == "Detectar bytes idénticos",
    "fps=1/N": q_sampling == "Muestreo temporal",
    "Trazabilidad": q_raw == "Conservar RAW y escribir derived aparte",
}
for name, ok in answers.items():
    print(("✓" if ok else "·"), name, "—", "correcto" if ok else "revisa la diapositiva asociada")

## Hoja de trucos

| Necesidad | Herramienta / dato |
|---|---|
| identidad exacta | SHA-256 |
| inspeccionar container/streams | `ffprobe -show_format -show_streams -of json` |
| audio canónico | FFmpeg · `-ac 1 -ar 16000` |
| muestrear video | FFmpeg · filtro `fps=1/N` |
| thumbnail | Pillow |
| conservar trazabilidad | RAW + derived + manifest |
| dataset interoperable | JSONL + Parquet |

## Puente a S11

La lógica ya funciona en un proceso. La siguiente pregunta es operacional:

> ¿Qué cambia cuando no son tres assets, sino millones?

S11 introduce particiones, scheduler, workers, lazy execution y shuffle usando este mismo modelo mental.